# Exploratory Data Analysis (EDA): Academic Labour Well-being (AT & CZ - Full Sample)

This notebook performs a detailed exploratory analysis on the **final processed dataset** (`labour_wellbeing_processed_v1.csv`) concerning the labour well-being of staff in Austria and the Czech Republic. The primary goal is to prepare the data and gain insights for subsequent Structural Equation Modeling (SEM) focused on explaining **Burnout_Score**.

**Objectives:**
1.  **Load and Prepare:** Load the processed data and apply necessary label encoding for interpretability, correctly identifying all categorical variables based on the final names from the preparation phase.
2.  **Univariate Analysis:** Understand the distribution and characteristics of *all* relevant variables using descriptive statistics and consistent visualizations (separating numerical and categorical).
3.  **Multivariate Analysis (Correlations):** Explore relationships using:
    * Scatter plot matrices (Pairplots) for key numerical variables (Work Conditions, Motivations (WM_), Attitudes (VB_)) related to `Burnout_Score`, with observations colored by `Country_Label`.
    * Full Pearson correlation matrix for all quantitative variables.
    * Grouped correlation matrices by theoretical domains focusing on the relationship with `Burnout_Score`.
4.  **Bivariate Analysis (Categorical vs. Numerical):** Systematically visualize how numerical variable distributions change across all categories of sociodemographic and work condition variables (using labeled categories).
5.  **Disparity Analysis (Compact):** Investigate potential differences in key well-being metrics (especially `Burnout_Score`) across specific demographic and institutional groups (e.g., `Country_Label`, `Gender_Label`, `Institution_Type_Label`, `Subject_Area_Label`) using grouped statistics and **compact subplot visualizations**.
6.  **SEM Preparation:** Identify potential multicollinearity issues and understand variable distributions to inform SEM model specification.

## 1. Setup: Import Libraries

In [1]:
# Core Libraries for Data Manipulation
import pandas as pd
import numpy as np
import math # For calculating subplot grid size

# Libraries for Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde # For smooth density lines in histograms

# Utilities
from IPython.display import display, Markdown # For displaying outputs nicely in Jupyter
import warnings # To manage warnings

# --- Configuration ---
warnings.filterwarnings('ignore') # Suppress routine warnings for cleaner output
sns.set_theme(style="whitegrid", palette="muted") # Set consistent plot theme
plt.rcParams['figure.figsize'] = (14, 6) # Default figure size
plt.rcParams['axes.titlesize'] = 16 # Title font size
plt.rcParams['axes.labelsize'] = 12 # Axis label font size
plt.rcParams['xtick.labelsize'] = 10 # X-tick label size
plt.rcParams['ytick.labelsize'] = 10 # Y-tick label size
plt.rcParams['figure.autolayout'] = True # Enable auto layout to prevent overlaps
pd.set_option('display.max_columns', None) # Show all columns in DataFrames
pd.set_option('display.width', 1000) # Adjust display width

## 2. Data Loading

Load the final processed dataset generated by `data_preparation.ipynb`.

In [2]:
# Specify the path to your final processed dataset
# This should be the output from the data_preparation notebook
file_path = 'https://raw.githubusercontent.com/EduardoAve/Labour-well-being/refs/heads/main/data/02_prepared/Final_Dataset_Processed.csv'

try:
    # Attempt to load the dataset
    df_raw = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' loaded successfully.")
    print(f"Dimensions: {df_raw.shape}")
    display(df_raw.head())
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Please ensure the file is in the correct directory or provide the full path.")
    df_raw = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    df_raw = pd.DataFrame()

An error occurred while loading the data: HTTP Error 404: Not Found


## 3. Initial Data Check (Post-Loading)

Verify data types, check for any remaining missing values (ideally none, except perhaps in 'structural NaN' columns if they weren't fully handled or if rows were kept), and get a basic overview.

In [3]:
if not df_raw.empty:
    print("\n--- General Information and Data Types ---")
    df_raw.info()

    print("\n--- Missing Values Summary ---")
    missing_values = df_raw.isnull().sum()
    missing_percentage = (missing_values / len(df_raw)) * 100
    missing_info = pd.DataFrame({'Count': missing_values, 'Percentage': missing_percentage})
    missing_info = missing_info[missing_info['Count'] > 0]
    if not missing_info.empty:
        print("Warning: Missing values found in the 'processed' dataset:")
        display(missing_info.sort_values(by='Count', ascending=False))
    else:
        print("No missing values found in the loaded dataset.")

    # Define conceptually categorical columns (using final names from data prep)
    # These are the columns that will be label-encoded
    categorical_original_names = [
        'Country', 'Gender', 'Marital_Status', 'Cares_for_Dependents',
        'Institution_Type', 'Subject_Area', 'Contract_Duration',
        'Effort_Comparison', 'Holds_Leadership_Position', 'Policy_Influence', 'Has_Other_Job',
        'Academic/Non-academic', 'Current_Position', 'Job_Description_Category',
        'Education_Level'
        # Add other columns that are numeric codes for categories if needed
    ]
    # Also include Version if it's just an ID and shouldn't be in numeric stats
    cols_to_exclude_from_numeric = ['Version'] + categorical_original_names

    print("\n--- Initial Descriptive Statistics (Numerical Variables) ---")
    # Ensure we only describe columns that are actually numeric
    numeric_cols_initial = df_raw.select_dtypes(include=np.number).columns
    # Exclude the columns that represent categories numerically
    numeric_cols_for_desc = numeric_cols_initial.difference(cols_to_exclude_from_numeric, sort=False)

    if not numeric_cols_for_desc.empty:
        display(df_raw[numeric_cols_for_desc].describe().T.round(2))
    else:
        print("No numerical columns identified for descriptive statistics after excluding categorical codes.")

    # Create a working copy for preprocessing. This will be our main analysis dataframe.
    df_analysis = df_raw.copy()

else:
    print("DataFrame is empty. Further analysis cannot proceed.")
    df_analysis = pd.DataFrame() # Ensure df_analysis exists even if empty

DataFrame is empty. Further analysis cannot proceed.


## 4. Data Preparation for EDA (Labeling)

Steps:
1.  **Label Encoding:** Convert numerical categorical variables to meaningful text labels based on the data dictionary and preparation script. Ensure all relevant categorical variables are correctly identified and mapped.
2.  **Create Age Groups:** Bin the 'Age' variable for grouped analysis.

### 4.1 Label Encoding for Categorical Variables

Apply mappings to create human-readable labels for categorical variables stored as numbers. Verify mappings against the data dictionary and the actual codes present in the data.

In [4]:
if not df_analysis.empty:
    print("Applying label encoding to categorical variables...")
    # --- Define Mappings (CRITICAL: Verify these match your actual data codes and desired labels) ---
    country_map = {1: 'Austria', 2: 'Czech Republic'}
    gender_map = {1.0: 'Male', 2.0: 'Female', 3.0: 'Other'}
    marital_map = {
        1.0: 'Married/Registered Partnership', 2.0: 'In a relationship (unmarried)',
        3.0: 'Single', 4.0: 'Divorced', 5.0: 'Widowed', 6.0: 'Other_Marital'
    }
    care_map = {
        1.0: 'No', 2.0: 'Care for underage children', 3.0: 'Care for dependent relatives',
        4.0: 'Combination Care'
    }
    institution_type_map = {
        1.0: 'CZ: Public HEI', 2.0: 'CZ: Private HEI', 3.0: 'CZ: State HEI',
        4.0: 'AT: Public University', 5.0: 'AT: Private University/College',
        6.0: 'AT: University of Applied Sciences',
        7.0: 'AT: Public Uni College Teacher Ed', 8.0: 'AT: Private Uni College Teacher Ed'
    }
    subject_area_map = {
        1.0: 'Natural sciences', 2.0: 'Technical sciences', 3.0: 'Agricultural/forestry/veterinary',
        4.0: 'Healthcare/medical/pharmaceutical', 5.0: 'Humanities/social sciences',
        6.0: 'Economic sciences', 7.0: 'Law', 8.0: 'Pedagogy/teacher training',
        9.0: 'Culture/art', 10.0: 'Sport sciences', 11.0: 'Unspecified/cannot be categorised',
        12.0: 'Security/defence/Military', 13.0: 'Other_Faculty'
    }
    contract_duration_map = {
        1.0: 'Permanent/Continuous (CZ/AT)', 2.0: 'Fixed-term (permanent prospects) (CZ/AT)',
        3.0: 'Fixed-term (no permanent prospects) (CZ/AT)', 4.0: 'Casual/hourly (CZ/AT)',
        5.0: 'Fixed-term (unspecified prospects AT?)', 6.0: 'Permanent (tenured AT?)',
        7.0: 'Other_Contract'
    }
    effort_comparison_map = {1.0: 'Equal', 2.0: 'Less', 3.0: 'More'}
    holds_leadership_map = {
        1.0: 'No', 2.0: 'Yes (Institution/Faculty/Dept)', 3.0: 'Yes (Research Team)',
        4.0: 'Combination Leadership'
    }
    policy_influence_map = {1.0: '1: Not influential', 2.0: '2', 3.0: '3', 4.0: '4', 5.0: '5: Very influential'}
    academic_non_academic_map = {1: 'Non-academic', 2: 'Academic'}
    current_position_map = {
        2.0: 'Lecturer (CZ/AT Lector)', 3.0: 'Assistant (CZ)', 4.0: 'Assistant professor (CZ)',
        5.0: 'Docent (CZ)', 6.0: 'Professor (CZ)', 7.0: 'Researcher (CZ)',
        8.0: 'Externist (CZ)', 9.0: 'Researcher & Academic (CZ)',
        10.0: 'Postdoc Assistant (AT)', 11.0: 'Assistant Prof (AT)', 12.0: 'Associate Prof (AT)',
        13.0: 'University Prof (AT)', 14.0: 'Senior Scientist/Lecturer (AT)', 15.0: 'Project Staff (AT)',
        16.0: 'Other Position (AT)', 17.0: 'Student Assistant (AT)',
        1.0: 'Non-academic Role (Generic)', 0.0: 'Unknown/NA Position'
    }
    job_description_map = {
        1.0: 'Dept/manager assistant', 2.0: 'Lab technician', 3.0: 'Librarian/archivist',
        4.0: 'Facility management', 5.0: 'ICT', 6.0: 'Student support',
        7.0: 'Economics/finance/HR', 8.0: 'Project management', 9.0: 'Legal/control',
        10.0: 'Marketing/PR', 11.0: 'Science/knowledge transfer', 12.0: 'Foreign affairs',
        13.0: 'Other administrative', 14.0: 'Other/Combination Admin'
    }
    education_level_map = {
        1.0: 'Elementary', 2.0: 'Apprenticeship', 3.0: 'Vocational/Commercial School',
        4.0: 'High school/Secondary', 5.0: 'Higher professional school (CZ)',
        6.0: 'Bachelor', 7.0: 'Master', 8.0: 'Doctoral/PhD'
    }
    has_other_job_map = {
        1.0: 'No', 2.0: 'Yes (Public Sector)', 3.0: 'Yes (Private Sector)',
        4.0: 'Yes (Non-profit)', 5.0: 'Yes (Self-employed)',
        6.0: 'Yes (Multiple Areas)', 7.0: 'Yes (Other_Job)'
    }

    mappings_to_apply = {
        'Country': country_map,
        'Gender': gender_map,
        'Marital_Status': marital_map,
        'Cares_for_Dependents': care_map,
        'Institution_Type': institution_type_map,
        'Subject_Area': subject_area_map,
        'Contract_Duration': contract_duration_map,
        'Effort_Comparison': effort_comparison_map,
        'Holds_Leadership_Position': holds_leadership_map,
        'Policy_Influence': policy_influence_map,
        'Academic/Non-academic': academic_non_academic_map,
        'Current_Position': current_position_map,
        'Job_Description_Category': job_description_map,
        'Education_Level': education_level_map,
        'Has_Other_Job': has_other_job_map
    }

    labeled_cols = []

    for col, mapping in mappings_to_apply.items():
        if col in df_analysis.columns:
            label_col = f"{col}_Label"
            df_analysis[label_col] = df_analysis[col].map(mapping)
            is_ordered = (col == 'Policy_Influence' or col == 'Effort_Comparison' or col == 'Education_Level')
            try:
                all_categories = sorted(list(set(mapping.values()))) if not is_ordered else list(mapping.values())
                df_analysis[label_col] = pd.Categorical(df_analysis[label_col], categories=all_categories, ordered=is_ordered)
                print(f"  - Labeled column '{label_col}' created for '{col}'. Type: {'Ordinal' if is_ordered else 'Nominal'}")
                labeled_cols.append(label_col)
            except Exception as e:
                print(f"  - Warning: Could not convert '{label_col}' to categorical. Error: {e}")
                if label_col in df_analysis.columns:
                    df_analysis[label_col] = df_analysis[label_col].astype('object')
        else:
            print(f"  - Warning: Column '{col}' not found in DataFrame, skipping mapping.")

    print("\nLabel encoding completed.")

    if labeled_cols:
        display(df_analysis[labeled_cols].head())
    print("\nData types after label encoding:")
    df_analysis.info()

else:
    print("DataFrame is empty, label encoding skipped.")


DataFrame is empty, label encoding skipped.


### 4.2 Create Age Groups

In [5]:
if not df_analysis.empty:
    age_bins = [0, 29.9, 39.9, 49.9, 59.9, np.inf]
    age_labels = ['<30', '30-39', '40-49', '50-59', '60+']
    if 'Age' in df_analysis.columns:
        df_analysis['Age_Group'] = pd.cut(df_analysis['Age'], bins=age_bins, labels=age_labels, right=True)
        df_analysis['Age_Group'] = pd.Categorical(df_analysis['Age_Group'], categories=age_labels, ordered=True)
        print("'Age_Group' column created successfully.")
        if 'labeled_cols' in locals() and 'Age_Group' not in labeled_cols:
            labeled_cols.append('Age_Group')
    else:
        print("Warning: 'Age' column not found, could not create 'Age_Group'.")
else:
    print("Empty DataFrame, age group creation skipped.")

Empty DataFrame, age group creation skipped.


## 5. Exploratory Data Analysis (EDA) - Full Sample

Analyze distributions, relationships, and potential disparities in the full processed data (`df_analysis`).

### 5.1 Univariate Analysis (Full Sample)

Examine the distribution of each variable.

In [6]:
if not df_analysis.empty:
    # --- Identify final numeric and categorical columns for analysis ---
    all_numeric_df_cols = df_analysis.select_dtypes(include=np.number).columns
    numeric_cols_for_analysis = all_numeric_df_cols.difference(cols_to_exclude_from_numeric, sort=False)
    categorical_cols_for_analysis = [col for col in df_analysis.columns if col.endswith('_Label') or col == 'Age_Group']

    # --- Numerical Descriptive Statistics ---
    display(Markdown("#### Descriptive Statistics (Numerical Variables - Full Sample)"))
    if not numeric_cols_for_analysis.empty:
        display(df_analysis[numeric_cols_for_analysis].describe().T.round(2))
    else:
        print("No numerical columns found for descriptive statistics.")

    # --- Categorical Frequencies ---
    display(Markdown("#### Frequencies (Categorical Variables - Labeled - Full Sample)"))
    if categorical_cols_for_analysis:
        for col in categorical_cols_for_analysis:
            if col in df_analysis.columns:
                display_col_name = col.replace('_Label','').replace('_',' ')
                display(Markdown(f"##### {display_col_name}"))
                freq_table = df_analysis[col].value_counts(dropna=False).to_frame(name="Frequency")
                freq_table['Percentage'] = (df_analysis[col].value_counts(normalize=True, dropna=False) * 100).round(2)
                display(freq_table)
            else:
                print(f"Warning: Column {col} not found for frequency count.")
    else:
        print("No labeled categorical columns found for frequency counts.")

else:
    print("DataFrame is empty, univariate statistics skipped.")

DataFrame is empty, univariate statistics skipped.


In [7]:
if not df_analysis.empty:
    display(Markdown("#### Univariate Visualizations (Full Sample)"))
    uni_palette = "viridis"
    fig_size_uni = (14, 5)

    # --- Plots for Numerical Variables ---
    if not numeric_cols_for_analysis.empty:
        display(Markdown("##### Numerical Distributions (Histograms & Boxplots - Full Sample)"))
        for col in numeric_cols_for_analysis:
            clean_col_name = col.replace('_', ' ')
            try:
                if df_analysis[col].notna().sum() < 2: continue
                fig, axes = plt.subplots(1, 2, figsize=fig_size_uni)
                fig.suptitle(f'Distribution of {clean_col_name} (Full Sample)', fontsize=16, y=1.03)
                sns.histplot(df_analysis[col], kde=True, ax=axes[0], bins=30, color=sns.color_palette(uni_palette, 2)[0])
                axes[0].set_title('Histogram & Density'); axes[0].set_xlabel(clean_col_name); axes[0].set_ylabel('Frequency / Density')
                sns.boxplot(x=df_analysis[col], ax=axes[1], color=sns.color_palette(uni_palette, 2)[1])
                axes[1].set_title('Boxplot'); axes[1].set_xlabel(clean_col_name)
                plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()
            except Exception as e:
                print(f"Could not plot numerical variable {col}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
    else:
        print("No numerical columns for plotting.")

    # --- Plots for Categorical Variables ---
    if categorical_cols_for_analysis:
        display(Markdown("##### Categorical Distributions (Bar Charts - Full Sample)"))
        for col in categorical_cols_for_analysis:
            if col in df_analysis.columns:
                clean_col_name = col.replace('_Label','').replace('_',' ')
                try:
                    num_categories = df_analysis[col].nunique(dropna=False)
                    if df_analysis[col].notna().any() and num_categories > 0:
                        dynamic_height = max(5, num_categories * 0.45)
                        plt.figure(figsize=(10, dynamic_height))
                        order = df_analysis[col].value_counts(dropna=False).index
                        ax = sns.countplot(y=df_analysis[col], order=order, palette=uni_palette)
                        ax.set_title(f'Distribution of {clean_col_name} (Full Sample)'); ax.set_xlabel('Frequency Count'); ax.set_ylabel('')
                        plt.yticks(fontsize=9); plt.tight_layout(); plt.show()
                    else: pass
                except Exception as e:
                    print(f"Could not plot categorical variable {col}. Error: {e}")
                    if plt.gcf().get_axes(): plt.close()
            else:
                print(f"Warning: Column {col} not found for plotting.")
    else:
        print("No labeled categorical columns for plotting.")
else:
    print("DataFrame is empty, univariate visualizations skipped.")

DataFrame is empty, univariate visualizations skipped.


### 5.2 Multivariate Analysis (Correlations & Pairplots - Full Sample)

Explore relationships between numerical variables in the full sample, using `Country_Label` for hue in pairplots.

#### 5.2.1 Scatter Plot Matrices (Pairplots - Full Sample)

In [8]:
if not df_analysis.empty:
    display(Markdown("##### Scatter Plot Matrices (Pairplots - Full Sample, Hue by Country)"))

    all_numeric_vars_for_plots = numeric_cols_for_analysis

    pairplot1_vars_candidates = [
        'Age', 'Burnout_Score', 'Job_Satisfaction', 'Salary/hour', 'Avg_Work_Hours_HE',
        'Perceived_Autonomy', 'Performance_Pressure', 'Academic_Resources',
        'Quality_Leadership', 'Sense_Community'
    ]
    pairplot1_vars = [var for var in pairplot1_vars_candidates if var in all_numeric_vars_for_plots]

    pairplot2_vars_candidates = [col for col in all_numeric_vars_for_plots if col.startswith('VB_')] + ['Burnout_Score']
    pairplot2_vars = [var for var in pairplot2_vars_candidates if var in all_numeric_vars_for_plots]

    pairplot3_vars_candidates = [col for col in all_numeric_vars_for_plots if col.startswith('WM_')] + ['Burnout_Score']
    pairplot3_vars = [var for var in pairplot3_vars_candidates if var in all_numeric_vars_for_plots]

    def generate_pairplot_general(data, vars_list, title_suffix, hue_col='Country_Label'):
        if len(vars_list) > 1:
            print(f"Generating pairplot for: {vars_list} ({len(vars_list)} variables), hue by {hue_col}")

            # Ensure hue column exists
            if hue_col not in data.columns:
                print(f"Warning: Hue column '{hue_col}' not found. Plotting without hue.")
                hue_col = None

            plot_data_cols = vars_list + ([hue_col] if hue_col else [])
            pairplot_data = data[plot_data_cols].copy()

            try:
                g = sns.pairplot(pairplot_data.dropna(subset=vars_list),
                               hue=hue_col,
                               diag_kind='kde',
                               plot_kws={'alpha': 0.4, 's': 30, 'edgecolor': None},
                               height=1.8)

                num_plot_vars = len(vars_list)
                for i in range(num_plot_vars):
                    if i < g.axes.shape[1]:
                       g.axes[num_plot_vars-1, i].set_xlabel(g.x_vars[i], rotation=45, ha='right', va='top', fontsize=8)
                for i in range(num_plot_vars):
                    if i < g.axes.shape[0]:
                        g.axes[i, 0].set_ylabel(g.y_vars[i], fontsize=8)

                plt.suptitle(f'Scatter Plot Matrix: {title_suffix} (Full Sample)', y=1.02, fontsize=14)
                g.fig.tight_layout(rect=[0, 0.02, 1, 0.98])
                plt.show()
            except Exception as e:
                print(f"Could not generate pairplot for {title_suffix}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
        else:
            print(f"Skipping pairplot for {title_suffix}: Not enough variables ({len(vars_list)} found).")

    display(Markdown("###### Pairplot 1: Work Conditions & Demographics vs. Burnout (Full Sample)"))
    generate_pairplot_general(df_analysis, pairplot1_vars, "Work Conditions & Demographics vs. Burnout")
    display(Markdown("###### Pairplot 2: Work Attitudes (VB_) vs. Burnout (Full Sample)"))
    generate_pairplot_general(df_analysis, pairplot2_vars, "Work Attitudes (VB_) vs. Burnout")
    display(Markdown("###### Pairplot 3: Work Motivations (WM_) vs. Burnout (Full Sample)"))
    generate_pairplot_general(df_analysis, pairplot3_vars, "Work Motivations (WM_) vs. Burnout")

else:
    print("DataFrame is empty, pairplots skipped.")

DataFrame is empty, pairplots skipped.


#### 5.2.2 Full Correlation Matrix (Pearson - Full Sample)

Visualize the linear relationships between all numerical variables in the full sample.

In [9]:
if not df_analysis.empty:
    display(Markdown("##### Full Correlation Matrix (Pearson - Full Sample)"))

    numeric_cols_for_corr = numeric_cols_for_analysis

    if not numeric_cols_for_corr.empty and len(numeric_cols_for_corr) > 1:
        correlation_matrix_full = df_analysis[numeric_cols_for_corr].corr(method='pearson')
        mask = np.triu(np.ones_like(correlation_matrix_full, dtype=bool))
        plt.figure(figsize=(max(12, len(numeric_cols_for_corr)*0.6), max(10, len(numeric_cols_for_corr)*0.5)))
        sns.heatmap(correlation_matrix_full,
                    mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
                    linewidths=.5, cbar_kws={"shrink": .7}, annot=False, fmt=".2f")
        plt.title('Full Correlation Matrix (Pearson - Full Sample)', fontsize=16)
        plt.xticks(rotation=60, ha='right', fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        plt.show()
    elif len(numeric_cols_for_corr) <= 1:
        print("Not enough quantitative columns (>1) to calculate correlation matrix.")
    else:
        print("No quantitative columns found for correlation analysis.")
else:
    print("DataFrame is empty, full correlation analysis skipped.")

DataFrame is empty, full correlation analysis skipped.


#### 5.2.3 Grouped Correlation Matrices (Full Sample)

Examine correlations within domains and between predictors and `Burnout_Score` for the full sample.

In [10]:
# Define the heatmap plotting function here, before it's called
def plot_correlation_heatmap(corr_matrix, title):
    """Helper function to plot a correlation heatmap."""
    if corr_matrix.empty or corr_matrix.shape[0] < 1 or corr_matrix.shape[1] < 1:
        print(f"Skipping heatmap for '{title}': Not enough variables or empty matrix.")
        return
    mask = None
    if corr_matrix.shape[0] == corr_matrix.shape[1] and corr_matrix.shape[0] > 1:
         mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    annot_size = 8 if max(corr_matrix.shape) < 15 else 7
    plt.figure(figsize=(max(8, corr_matrix.shape[1]*0.8), max(6, corr_matrix.shape[0]*0.6)))
    sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
                linewidths=.5, cbar_kws={"shrink": .7}, annot=True, fmt=".2f", annot_kws={"size": annot_size})
    plt.title(title, fontsize=14); plt.xticks(rotation=45, ha='right', fontsize=9); plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout(); plt.show()

if not df_analysis.empty:
    display(Markdown("##### Grouped Correlation Matrices (Full Sample)"))
    all_numeric_vars_corr = numeric_cols_for_analysis

    demographic_vars_num = [v for v in ['Age'] if v in all_numeric_vars_corr]
    work_conditions_vars_num = [v for v in ['Avg_Work_Hours_HE', 'Salary/hour', 'Salary effort/hour',
                                           'Avg_Work_Hours_Other', 'Teaching %', 'Research %',
                                           'Activities related to externally funded research projects %',
                                           'Organisational and administrative activities %'] if v in all_numeric_vars_corr]
    subjective_perception_vars = [v for v in ['Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy',
                                               'Quality_Leadership', 'Sense_Community'] if v in all_numeric_vars_corr]
    motivation_vars = [v for v in all_numeric_vars_corr if v.startswith('WM_')]
    attitudes_vars = [v for v in all_numeric_vars_corr if v.startswith('VB_')]
    wellbeing_outcomes = [v for v in ['Job_Satisfaction', 'Burnout_Score', 'Vulnerability'] if v in all_numeric_vars_corr]

    if 'correlation_matrix_full' not in locals() or correlation_matrix_full.empty:
        if not all_numeric_vars_corr.empty and len(all_numeric_vars_corr) > 1:
            correlation_matrix_full = df_analysis[all_numeric_vars_corr].corr(method='pearson')
        else:
            correlation_matrix_full = pd.DataFrame()

    if not correlation_matrix_full.empty:
        groups_to_plot_intra = {
            "Work Conditions (Numeric)": work_conditions_vars_num,
            "Subjective Perceptions": subjective_perception_vars,
            "Motivations (WM_)": motivation_vars,
            "Attitudes (VB_)": attitudes_vars,
            "Well-being Outcomes": wellbeing_outcomes
        }
        for group_name, group_vars in groups_to_plot_intra.items():
            valid_group_vars = [var for var in group_vars if var in all_numeric_vars_corr]
            if len(valid_group_vars) > 1:
                group_corr = correlation_matrix_full.loc[valid_group_vars, valid_group_vars]
                plot_correlation_heatmap(group_corr, f'Intra-Group Correlation: {group_name} (Full Sample)')
            else:
                print(f"Skipping intra-group heatmap for '{group_name}': Not enough valid variables.")

        key_outcome_burnout = ['Burnout_Score']
        valid_key_outcome_burnout = [var for var in key_outcome_burnout if var in all_numeric_vars_corr]
        predictor_groups_numeric = {
            "Demographics & Work Conditions": demographic_vars_num + work_conditions_vars_num,
            "Subjective Perceptions": subjective_perception_vars,
            "Motivations (WM_)": motivation_vars,
            "Attitudes (VB_)": attitudes_vars
        }
        if valid_key_outcome_burnout:
            display(Markdown(f"#### Correlations between Predictor Groups and {valid_key_outcome_burnout[0]} (Full Sample)"))
            for group_name, pred_vars in predictor_groups_numeric.items():
                valid_pred_vars = [var for var in pred_vars if var in all_numeric_vars_corr]
                if valid_pred_vars and valid_key_outcome_burnout[0] in correlation_matrix_full.columns:
                    cross_corr = correlation_matrix_full.loc[valid_pred_vars, valid_key_outcome_burnout]
                    if not cross_corr.empty: plot_correlation_heatmap(cross_corr, f'Correlations: {group_name} vs. {valid_key_outcome_burnout[0]} (Full Sample)')
                    else: print(f"No valid correlations for '{group_name}' vs. {valid_key_outcome_burnout[0]}.")
                else: print(f"Skipping cross-correlation for '{group_name}': Invalid predictors or outcome.")
        else: print(f"Key outcome variable ({key_outcome_burnout[0]}) not found or not numeric.")
    else: print("Full correlation matrix is empty.")
else: print("DataFrame is empty, grouped correlation analysis skipped.")

DataFrame is empty, grouped correlation analysis skipped.


### 5.3 Bivariate Analysis: Categorical vs. Numerical Variables (Full Sample)

Visualize distributions of numerical variables across categories in the full sample.

In [11]:
if 'df_analysis' in locals() and not df_analysis.empty:
    numeric_cols_for_biv_plots = numeric_cols_for_analysis
    categorical_cols_for_biv_plots = categorical_cols_for_analysis
    biv_palette = "pastel"

    if not numeric_cols_for_biv_plots.empty and categorical_cols_for_biv_plots:
        display(Markdown("##### Comparison of Numerical Distributions by Category (Full Sample)"))
        for cat_col in categorical_cols_for_biv_plots:
            if cat_col in df_analysis.columns:
                clean_cat_col_name = cat_col.replace('_Label','').replace('_',' ')
                display(Markdown(f"###### Comparisons by: {clean_cat_col_name}"))
                for num_col in numeric_cols_for_biv_plots:
                    clean_num_col_name = num_col.replace('_', ' ')
                    try:
                        num_categories = df_analysis[cat_col].nunique(dropna=False)
                        if df_analysis[[cat_col, num_col]].dropna().empty or num_categories == 0: continue
                        max_label_len = 0
                        if df_analysis[cat_col].dtype == 'category' and df_analysis[cat_col].cat.categories.inferred_type == 'string':
                           max_label_len = df_analysis[cat_col].cat.categories.str.len().max()
                        elif df_analysis[cat_col].dtype == 'object' and df_analysis[cat_col].dropna().apply(lambda x: isinstance(x, str)).all():
                           max_label_len = df_analysis[cat_col].str.len().max()
                        base_width_per_cat = 0.8
                        if max_label_len > 15: base_width_per_cat = max_label_len * 0.08
                        fig_width = max(10, num_categories * base_width_per_cat)
                        plt.figure(figsize=(fig_width, 6))
                        plot_order = None
                        if isinstance(df_analysis[cat_col].dtype, pd.CategoricalDtype):
                            if df_analysis[cat_col].cat.ordered: plot_order = df_analysis[cat_col].cat.categories.tolist()
                            elif num_categories < 15: plot_order = df_analysis[cat_col].value_counts().index
                        ax = sns.violinplot(x=cat_col, y=num_col, data=df_analysis, palette=biv_palette, cut=0, inner='quartile', order=plot_order)
                        ax.set_title(f'{clean_num_col_name} by {clean_cat_col_name} (Full Sample)'); ax.set_xlabel(clean_cat_col_name); ax.set_ylabel(clean_num_col_name)
                        if num_categories > 4 or max_label_len > 10: plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                        else: plt.setp(ax.get_xticklabels(), rotation=0)
                        plt.tight_layout(); plt.show()
                    except Exception as e:
                        print(f"Could not plot {num_col} by {cat_col}. Error: {e}")
                        if plt.gcf().get_axes(): plt.close()
            else: print(f"Warning: Categorical column {cat_col} not found.")
    else: print("Not enough numerical or categorical variables for bivariate analysis.")
else: print("DataFrame is empty, Bivariate Categorical vs Numerical analysis skipped.")

DataFrame is empty, Bivariate Categorical vs Numerical analysis skipped.


### 5.4 Disparity Analysis (Compact Subplots - Full Sample)

Explore how key well-being variables differ across specific demographic and institutional groups in the full sample. Use subplots to present comparisons more compactly.

In [12]:
if 'df_analysis' in locals() and not df_analysis.empty:
    grouping_vars_labels = categorical_cols_for_analysis
    target_vars_grouped_candidates = [
        'Burnout_Score', 'Job_Satisfaction', 'Perceived_Autonomy', 'Performance_Pressure',
        'Academic_Resources', 'WM_Intrinsic_Motivation', 'VB_Emotional_Distancing', 'Salary/hour'
    ]
    valid_target_vars_grouped = [var for var in target_vars_grouped_candidates if var in numeric_cols_for_analysis]
    print(f"Key variables selected for compact disparity analysis: {valid_target_vars_grouped}")
    grouped_palette = "muted"; num_targets = len(valid_target_vars_grouped)

    if grouping_vars_labels and valid_target_vars_grouped:
        for group_var in grouping_vars_labels:
            if group_var in df_analysis.columns and df_analysis[group_var].notna().any() and df_analysis[group_var].nunique() > 1:
                clean_group_var_name = group_var.replace('_Label','').replace('_',' ')
                display(Markdown(f"#### Disparity Analysis by: {clean_group_var_name} (Full Sample)"))
                try:
                    use_observed = pd.__version__ >= '1.5.0' and pd.api.types.is_categorical_dtype(df_analysis[group_var])
                    grouped_stats = df_analysis.groupby(group_var, observed=use_observed)[valid_target_vars_grouped].agg(['mean', 'median'])
                    display(grouped_stats.round(2))
                except Exception as e: print(f"Could not calculate grouped stats for {group_var}. Error: {e}"); continue

                num_categories = df_analysis[group_var].nunique(dropna=False)
                if num_categories == 0: continue
                ncols = 2 if num_targets > 1 else 1; nrows = math.ceil(num_targets / ncols)
                max_cat_label_len = 0
                if df_analysis[group_var].dtype == 'category' and df_analysis[group_var].cat.categories.inferred_type == 'string':
                   max_cat_label_len = df_analysis[group_var].cat.categories.str.len().max()
                elif df_analysis[group_var].dtype == 'object' and df_analysis[group_var].dropna().apply(lambda x: isinstance(x, str)).all():
                   max_cat_label_len = df_analysis[group_var].str.len().max()
                base_width_per_cat_subplot = 0.8
                if max_cat_label_len > 15: base_width_per_cat_subplot = max_cat_label_len * 0.07
                elif max_cat_label_len > 10: base_width_per_cat_subplot = max_cat_label_len * 0.09
                fig_height_subplot = 5 * nrows; fig_width_subplot = max(10, num_categories * base_width_per_cat_subplot) * ncols
                fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width_subplot, fig_height_subplot), squeeze=False); axes = axes.flatten()
                plot_order = None
                if isinstance(df_analysis[group_var].dtype, pd.CategoricalDtype):
                     if df_analysis[group_var].cat.ordered: plot_order = df_analysis[group_var].cat.categories.tolist()
                     elif num_categories < 15: plot_order = df_analysis[group_var].value_counts().index

                for i, target_var in enumerate(valid_target_vars_grouped):
                    ax = axes[i]; clean_target_var_name = target_var.replace('_',' ')
                    try:
                        sns.violinplot(x=group_var, y=target_var, data=df_analysis, palette=grouped_palette, cut=0, inner='quartile', order=plot_order, ax=ax)
                        ax.set_title(f'{clean_target_var_name}'); ax.set_xlabel(''); ax.set_ylabel(clean_target_var_name)
                        if num_categories > 4 or max_cat_label_len > 10: plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                        else: plt.setp(ax.get_xticklabels(), rotation=0)
                    except Exception as e: print(f"Could not plot {target_var} by {group_var} on subplot. Error: {e}"); ax.set_title(f'{clean_target_var_name} (Plot Error)')
                for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
                fig.suptitle(f'Distribution of Key Metrics by {clean_group_var_name} (Full Sample)', fontsize=16, y=1.02)
                plt.tight_layout(rect=[0, 0, 1, 0.98]); plt.show()
            else: pass
    else: print("Could not perform disparity analysis: Missing grouping or target variables.")
else: print("DataFrame is empty, disparity analysis skipped.")

DataFrame is empty, disparity analysis skipped.


## 6. Preliminary EDA Conclusions and Next Steps for SEM (Full Sample)

This exploratory analysis of the full processed dataset provides crucial groundwork for subsequent Structural Equation Modeling (SEM) aimed at explaining `Burnout_Score`.

**Key Observations Relevant to SEM (Full Sample):**
* **Variable Distributions:** The univariate analysis revealed the distributions of potential predictors and outcomes. Non-normal distributions might require robust estimation methods in SEM or variable transformations.
* **Correlations & Relationships:** Grouped correlation matrices and pairplots (colored by `Country_Label`) highlighted potential relationships within construct domains and between predictor domains and `Burnout_Score`. These inform initial SEM path specifications.
* **Country Differences:** The `hue='Country_Label'` in pairplots and specific disparity analyses by country (if `Country_Label` is used as a grouping variable) can reveal if relationships or mean levels of key variables differ significantly between Austria and the Czech Republic. This is crucial for deciding on multi-group SEM or including country as a covariate/moderator.
* **Potential Multicollinearity:** Strong correlations within predictor groups should be noted for potential multicollinearity issues.
* **Group Differences (General):** Disparity analyses across various demographic and institutional groups (Gender, Age Group, Institution Type, Subject Area) suggest these factors might be important covariates or moderators.
* **Key Predictor Areas:** Subjective perceptions, work attitudes (VB_), and motivations (WM_) appear correlated with `Burnout_Score`, reinforcing their importance.

**Next Steps Towards SEM:**
1.  **Theoretical Model Specification:** Define the hypothesized SEM based on theory and EDA findings.
2.  **Data Preparation for SEM Software:** Select columns, encode categoricals, scale/center if needed, and handle any final missing values.
3.  **SEM Estimation:** Estimate the model using appropriate software and estimators.
4.  **Model Evaluation:** Assess model fit and path coefficients.
5.  **Model Modification:** Refine the model based on fit and theory.
6.  **Mediation/Moderation Testing:** Formally test indirect and interaction effects.
7.  **Multi-Group Analysis:** If country differences are prominent, conduct multi-group SEM to compare model structures and parameters between Austria and the Czech Republic, including tests for measurement invariance.